In [1]:
"""
Dataset Formatter for Gemma 12B Instruction Tuning
====================================================
Handles two document types:
  1. Grammar explanations (📌 ব্যাখ্যা blocks)
  2. GK facts (Chapter/Cluster bullet lists)

Output: train.jsonl using Gemma chat template
"""

import re
import json
from pathlib import Path
from collections import defaultdict


# ─── Gemma 12B chat template ──────────────────────────────────────────────────

def gemma_format(user_content: str, assistant_content: str) -> str:
    return json.dumps({
        "text": (
            f"<start_of_turn>user\n{user_content}<end_of_turn>\n"
            f"<start_of_turn>model\n{assistant_content}<end_of_turn>"
        )
    }, ensure_ascii=False)


# ─── Document 1: Grammar explanation blocks ──────────────────────────────────

GRAMMAR_TOPICS = [
    "বিরোধ যোজক", "কারণ যোজক", "সাধারণ যোজক",
    "বিকল্প যোজক", "সাপেক্ষ যোজক", "যোজক",
    "সম্বোধন আবেগ", "অলংকার আবেগ", "বিরক্তি আবেগ",
    "বিস্ময় আবেগ", "আতঙ্ক আবেগ", "করুণা আবেগ", "আবেগ",
    "ক্রিয়াবিশেষণ বর্গ", "বিশেষ্য বর্গ", "বিশেষণ বর্গ", "ক্রিয়াবর্গ", "বর্গ",
    "পদাণু ক্রিয়াবিশেষণ", "ক্রিয়াবিশেষণ",
]

def infer_topic(text: str) -> str:
    for topic in GRAMMAR_TOPICS:
        if topic in text:
            return topic
    return "বাংলা ব্যাকরণ"

def clean_grammar_block(text: str) -> str:
    text = re.sub(r'📌 ব্যাখ্যা \d+-\d+:\s*', '', text)
    text = re.sub(r'উৎস:.*', '', text, flags=re.DOTALL)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def extract_example_question(block_text: str, topic: str) -> str:
    """Pull the quoted example from the block as the question."""
    quote = re.search(r"['\u2018\u2019''](.+?)['\u2018\u2019'']", block_text)
    if quote:
        example = quote.group(1).strip()
        return f"'{example}' — এই বাক্যে কোন ব্যাকরণিক উপাদান ব্যবহৃত হয়েছে? বিস্তারিত ব্যাখ্যা করো।"
    return f"{topic} কী? উদাহরণসহ ব্যাখ্যা করো।"

def parse_grammar_doc(text: str) -> list[dict]:
    """Parse document with 📌 ব্যাখ্যা blocks."""
    blocks = re.split(r'─{10,}', text)
    records = []

    for block in blocks:
        block = block.strip()
        if '📌' not in block:
            continue

        id_match = re.search(r'📌 ব্যাখ্যা (\d+-\d+)', block)
        entry_id = id_match.group(1) if id_match else "unknown"

        topic = infer_topic(block)
        cleaned = clean_grammar_block(block)

        if len(cleaned) < 50:
            continue

        question = extract_example_question(block, topic)

        records.append({
            "id": entry_id,
            "topic": topic,
            "question": question,
            "answer": cleaned,
        })

    return records


def grammar_to_jsonl(records: list[dict]) -> list[str]:
    """
    Group by topic so the model learns full patterns.
    Each topic cluster → one instruction example with full context.
    Individual examples → separate Q&A pairs.
    """
    lines = []

    # Group by topic
    topic_groups = defaultdict(list)
    for r in records:
        topic_groups[r["topic"]].append(r)

    for topic, group in topic_groups.items():
        # 1. Topic overview: all examples bundled
        all_answers = "\n\n---\n\n".join(r["answer"] for r in group)
        overview_q = f"{topic} সম্পর্কে বিস্তারিত ব্যাখ্যা করো এবং উদাহরণ দাও।"
        lines.append(gemma_format(overview_q, all_answers))

        # 2. Individual Q&A for each block
        for r in group:
            lines.append(gemma_format(r["question"], r["answer"]))

    return lines


# ─── Document 2: GK chapter/cluster facts ────────────────────────────────────

# Topic keyword mapping — order matters (more specific first)
GK_TOPIC_MAP = [
    ("গ্যাসক্ষেত্র ও তেলক্ষেত্র",   ["গ্যাসক্ষেত্র", "তেলক্ষেত্র", "হরিপুর", "বিবিয়ানা", "সাঙ্গু", "তিতাস", "গ্যাস উৎপাদন", "পেট্রোবাংলা"]),
    ("কয়লাক্ষেত্র",                 ["কয়লা", "বড়পুকুরিয়া", "জামালগঞ্জ", "দীঘিপাড়া", "খালাশপীর", "গন্ডোয়ানা"]),
    ("বিদ্যুৎ ও জ্বালানি",           ["বিদ্যুৎ", "কাপ্তাই", "পারমাণবিক", "বায়ুশক্তি", "রূপপুর", "মেগাওয়াট"]),
    ("রেলওয়ে ও যোগাযোগ",           ["রেলওয়ে", "রেলপথ", "রেলষ্টেশন", "ইস্টার্ন বেঙ্গল", "গ্র্যান্ড ট্রাঙ্ক"]),
    ("সুন্দরবন ও রামসার",           ["সুন্দরবন", "রামসার", "ম্যানগ্রোভ", "টাঙ্গুয়ার"]),
    ("চা শিল্প",                    ["চা", "চা বাগান", "চা গবেষণা", "মালনীছড়া"]),
    ("সার কারখানা",                  ["সার", "ফার্টিলাইজার", "ইউরিয়া", "কাফকো", "ফেঞ্চুগঞ্জ", "আশুগঞ্জ"]),
    ("বন্দর ও নৌপথ",                ["বন্দর", "নৌবন্দর", "খাল", "পানামা", "সুয়েজ", "কর্ণফুলী"]),
    ("আন্তর্জাতিক সংগঠন",           ["সার্ক", "ইউনেস্কো", "বিশ্বব্যাংক", "রামসার কনভেনশন", "ইউএন", "ইউনাইটেড নেশন"]),
    ("ঐতিহাসিক স্থান ও ঐতিহ্য",      ["ষাট গম্বুজ", "পাহাড়পুর", "ময়নামতি", "আলতামিরা", "বিশ্ব ঐতিহ্য"]),
    ("জলবায়ু ও পরিবেশ",             ["জলবায়ু", "পরিবেশ", "ঘূর্ণিঝড়", "ভূমিকম্প", "রামসার", "ইসিএ"]),
    ("নদী ও জলাশয়",                ["নদী", "হাওর", "ফারাক্কা", "তিস্তা", "কাপ্তাই বাঁধ", "কর্ণফুলী"]),
    ("খনিজ সম্পদ",                  ["চুনাপাথর", "কঠিন শিলা", "আর্সেনিক", "লোহা", "অ্যালুমিনিয়াম"]),
    ("কৃষি ও মৎস্য",                ["কৃষি", "মৎস্য", "চাষ", "বীজ", "কৃষিশুমারি"]),
    ("ভূগোল ও অবস্থান",             ["মরুভূমি", "পর্বত", "দ্বীপ", "সমুদ্র", "উপকূল", "মহীসোপান", "ভূমিরূপ"]),
    ("বিজ্ঞান ও প্রযুক্তি",         ["জিআইএস", "ইকোলজি", "করিওলিস", "জিএমটি", "ইউটিসি"]),
    ("শিল্প ও কারখানা",             ["কারখানা", "মিল", "শিল্প", "সিমেন্ট", "ইস্পাত", "নিউজপ্রিন্ট"]),
]

def infer_gk_topic(text: str) -> str:
    for topic, keywords in GK_TOPIC_MAP:
        if any(kw in text for kw in keywords):
            return topic
    return "সাধারণ জ্ঞান"

def parse_gk_doc(text: str) -> list[dict]:
    """Parse chapter/cluster structured GK facts."""
    chapter_pattern = re.compile(
        r'═{10,}.*?Chapter\s*/\s*Cluster\s*(\d+).*?═{10,}',
        re.DOTALL | re.IGNORECASE
    )
    parts = chapter_pattern.split(text)

    records = []
    chapter_num = 0

    # parts alternates: [pre_text, ch_num, ch_content, ch_num, ch_content, ...]
    i = 0
    while i < len(parts):
        if i + 2 <= len(parts) - 1 and parts[i + 1 if i + 1 < len(parts) else i].strip().isdigit():
            chapter_num = int(parts[i + 1])
            chapter_content = parts[i + 2] if i + 2 < len(parts) else ""
            i += 3
        elif i == 0 and chapter_num == 0:
            chapter_content = parts[0]
            chapter_num = 1
            i += 1
        else:
            i += 1
            continue

        bullets = re.findall(r'•\s+(.+?)(?=\n\s*•|\Z)', chapter_content, re.DOTALL)
        bullets = [b.replace('\n', ' ').strip() for b in bullets if len(b.strip()) > 20]

        if not bullets:
            continue

        # Bucket by topic
        topic_buckets = defaultdict(list)
        for b in bullets:
            topic = infer_gk_topic(b)
            topic_buckets[topic].append(b)

        for topic, facts in topic_buckets.items():
            if not facts:
                continue
            records.append({
                "chapter": chapter_num,
                "topic": topic,
                "facts": facts,
            })

    return records


def gk_record_to_instructions(record: dict) -> list[str]:
    """Generate 3 instruction types per topic-chunk."""
    lines = []
    topic = record["topic"]
    facts = record["facts"]
    facts_text = "\n".join(f"• {f}" for f in facts)

    # 1. Direct factual recall
    q1 = f"{topic} সম্পর্কে গুরুত্বপূর্ণ তথ্য ও তারিখগুলো উল্লেখ করো।"
    lines.append(gemma_format(q1, facts_text))

    # 2. For each fact, make an individual Q from the date/year mention
    for fact in facts:
        year_match = re.search(r'(\d{4})', fact)
        if year_match:
            year = year_match.group(1)
            q = f"{year} সালে {topic} সংক্রান্ত কী ঘটনা ঘটেছিল?"
            lines.append(gemma_format(q, f"• {fact}"))
        # Entity-based Q
        entity_match = re.search(r'বাংলাদেশ(?:ের)?|সুন্দরবন|কাপ্তাই|হরিপুর|সার্ক|ইউনেস্কো', fact)
        if entity_match:
            entity = entity_match.group(0)
            q = f"{entity} সম্পর্কিত এই তথ্যটি কী?"
            lines.append(gemma_format(q, f"• {fact}"))

    return lines


def gk_to_jsonl(records: list[dict]) -> list[str]:
    lines = []
    for r in records:
        lines.extend(gk_record_to_instructions(r))
    return lines


# ─── Deduplication ───────────────────────────────────────────────────────────

def deduplicate(lines: list[str], similarity_threshold: int = 0) -> list[str]:
    """
    Exact dedup on the full JSON string.
    For fuzzy dedup install: pip install datasketch
    """
    seen = set()
    unique = []
    for line in lines:
        if line not in seen:
            seen.add(line)
            unique.append(line)
    return unique


# ─── Main pipeline ────────────────────────────────────────────────────────────

def format_grammar_file(input_path: str, output_path: str):
    text = Path(input_path).read_text(encoding='utf-8')
    records = parse_grammar_doc(text)
    print(f"[Grammar] Parsed {len(records)} blocks")

    lines = grammar_to_jsonl(records)
    lines = deduplicate(lines)
    print(f"[Grammar] {len(lines)} unique instruction pairs")

    with open(output_path, 'w', encoding='utf-8') as f:
        for line in lines:
            f.write(line + '\n')
    print(f"[Grammar] Saved → {output_path}")
    return lines


def format_gk_file(input_path: str, output_path: str):
    text = Path(input_path).read_text(encoding='utf-8')
    records = parse_gk_doc(text)
    print(f"[GK] Parsed {len(records)} topic-chunks across chapters")

    lines = gk_to_jsonl(records)
    lines = deduplicate(lines)
    print(f"[GK] {len(lines)} unique instruction pairs")

    with open(output_path, 'w', encoding='utf-8') as f:
        for line in lines:
            f.write(line + '\n')
    print(f"[GK] Saved → {output_path}")
    return lines


def merge_and_shuffle(grammar_jsonl: str, gk_jsonl: str, output_path: str, seed: int = 42):
    import random
    random.seed(seed)

    all_lines = []
    for path in [grammar_jsonl, gk_jsonl]:
        if Path(path).exists():
            all_lines.extend(Path(path).read_text(encoding='utf-8').splitlines())

    random.shuffle(all_lines)

    with open(output_path, 'w', encoding='utf-8') as f:
        for line in all_lines:
            if line.strip():
                f.write(line + '\n')

    print(f"[Merge] {len(all_lines)} total examples → {output_path}")
    return all_lines


def validate_jsonl(path: str, n_samples: int = 3):
    """Print sample records to verify format."""
    print(f"\n{'='*60}")
    print(f"Validating {path}")
    print('='*60)
    lines = Path(path).read_text(encoding='utf-8').splitlines()
    print(f"Total records: {len(lines)}")

    import random
    samples = random.sample(lines, min(n_samples, len(lines)))
    for i, s in enumerate(samples, 1):
        obj = json.loads(s)
        print(f"\n--- Sample {i} ---")
        # Show first 300 chars of text
        print(obj["text"][:300] + "...")


# ─── Colab-ready usage ───────────────────────────────────────────────────────

if __name__ == "__main__":
    import sys

    # ── CHANGE THESE PATHS ──
    GRAMMAR_INPUT  = "/content/grammar_explanations.txt"   # your doc 1
    GK_INPUT       = "/content/data_tag_id_50_Stage4_final_output.txt"  # your doc 2

    GRAMMAR_JSONL  = "/content/grammar_train.jsonl"
    GK_JSONL       = "/content/gk_train.jsonl"
    MERGED_JSONL   = "/content/train.jsonl"
    # ────────────────────────

    grammar_lines, gk_lines = [], []

    if Path(GRAMMAR_INPUT).exists():
        grammar_lines = format_grammar_file(GRAMMAR_INPUT, GRAMMAR_JSONL)
    else:
        print(f"[Skip] Grammar file not found: {GRAMMAR_INPUT}")

    if Path(GK_INPUT).exists():
        gk_lines = format_gk_file(GK_INPUT, GK_JSONL)
    else:
        print(f"[Skip] GK file not found: {GK_INPUT}")

    if grammar_lines or gk_lines:
        merge_and_shuffle(GRAMMAR_JSONL, GK_JSONL, MERGED_JSONL)
        validate_jsonl(MERGED_JSONL)

    print("\nDone! Use train.jsonl with SFTTrainer.")
    print("Dataset stats:")
    if Path(MERGED_JSONL).exists():
        total = len(Path(MERGED_JSONL).read_text().splitlines())
        print(f"  Total examples : {total}")
        print(f"  Grammar approx : {len(grammar_lines)}")
        print(f"  GK approx      : {len(gk_lines)}")

[Skip] Grammar file not found: /content/grammar_explanations.txt
[GK] Parsed 31 topic-chunks across chapters
[GK] 1186 unique instruction pairs
[GK] Saved → /content/gk_train.jsonl
[Merge] 1186 total examples → /content/train.jsonl

Validating /content/train.jsonl
Total records: 1186

--- Sample 1 ---
<start_of_turn>user
বাংলাদেশের সম্পর্কিত এই তথ্যটি কী?<end_of_turn>
<start_of_turn>model
• মংলা বন্দর বাংলাদেশের দ্বিতীয় আন্তর্জাতিক সমুদ্র বন্দর।<end_of_turn>...

--- Sample 2 ---
<start_of_turn>user
বাংলাদেশের সম্পর্কিত এই তথ্যটি কী?<end_of_turn>
<start_of_turn>model
• বাংলাদেশের সর্ব দক্ষিণের জেলা হলো কক্সবাজার।<end_of_turn>...

--- Sample 3 ---
<start_of_turn>user
সুন্দরবন সম্পর্কিত এই তথ্যটি কী?<end_of_turn>
<start_of_turn>model
• সুন্দরবনের অধিকাংশ উদ্ভিদ চিরসবুজ হওয়ায় এদের শারীরবৃত্তিক ও গঠনগত অভিযোজন প্রায় একই রকম।<end_of_turn>...

Done! Use train.jsonl with SFTTrainer.
Dataset stats:
  Total examples : 1186
  Grammar approx : 0
  GK approx      : 1186
